In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from networkx.algorithms import community
import math
import warnings
import os

warnings.filterwarnings('ignore')

# ================= 配置区 =================
# 读取带有类型的详细图谱数据
FULL_KG_PATH = "AIBL_DementiaHKG_PrimeKG_Sub.csv"  
OUTPUT_PDF_PATH = "AIBL-DementiaHKGE.pdf"

def plot_static_kg_pdf_with_stats():
    print("==================================================")
    print("📊 开始统计 AIBL-DementiaHKG 图谱数据并绘制静态高清 PDF...")
    
    if not os.path.exists(FULL_KG_PATH):
        print(f"❌ 找不到全量图谱文件: {FULL_KG_PATH}，请确保路径正确。")
        return
        
    df = pd.read_csv(FULL_KG_PATH)
    
    # ---------------- 1. 数据统计 ----------------
    # 提取三元组总数
    num_triplets = len(df)
    
    # 提取所有去重后的节点及其类型
    nodes_x = df[['x_name', 'x_type']].rename(columns={'x_name': 'node', 'x_type': 'type'})
    nodes_y = df[['y_name', 'y_type']].rename(columns={'y_name': 'node', 'y_type': 'type'})
    all_nodes_df = pd.concat([nodes_x, nodes_y]).drop_duplicates(subset=['node'])
    
    num_nodes = len(all_nodes_df)
    
    # 构建有向图以获取真实的图谱边数
    G = nx.from_pandas_edgelist(df, source='x_name', target='y_name', edge_attr='relation', create_using=nx.DiGraph())
    num_edges = G.number_of_edges()
    
    # 统计每种节点类型的数量
    node_type_counts = all_nodes_df['type'].value_counts()
    
    # 统计每种关系类型的数量
    relation_counts = df['relation'].value_counts()
    
    # ---------------- 2. 控制台打印 ----------------
    print(f"\n📈 [整体规模统计]:")
    print(f"   - 节点总数: {num_nodes}")
    print(f"   - 边总数: {num_edges}")
    print(f"   - 三元组总数: {num_triplets}")
    
    print(f"\n🧬 [各类节点数量统计]:")
    for ntype, count in node_type_counts.items():
        print(f"   - {ntype}: {count}")
        
    print(f"\n🔗 [各类关系数量统计]:")
    for rel, count in relation_counts.items():
        print(f"   - {rel}: {count}")
        
    # ---------------- 3. 图谱绘制 (脉络化社区布局) ----------------
    print("\n⏳ 正在进行脉络化社区布局计算 (节点较多，可能需要 1-2 分钟，请耐心等待)...")
    
    # 记录每个节点的类型用于映射颜色
    node_types = dict(zip(all_nodes_df['node'], all_nodes_df['type']))
        
    # 颜色映射字典
    color_map = {
        'disease': '#ff4757',            
        'effect/phenotype': '#ffa502',   
        'gene/protein': '#2ed573',       
        'anatomy': '#1e90ff',            
        'biological_process': '#ff6b81', 
        'pathway': '#9c88ff',            
        'cellular_component': '#7bed9f', 
        'molecular_function': '#70a1ff'  
    }
    
    # 开始渲染 Matplotlib 图像
    plt.figure(figsize=(24, 24), facecolor='#1a1a1a') # 使用深色背景提升对比度
    ax = plt.gca()
    ax.set_facecolor('#1a1a1a')
    
    # 进行社区检测 (使用无向图版本以适配算法)
    G_undirected = G.to_undirected()
    communities_generator = community.greedy_modularity_communities(G_undirected)
    communities = sorted(communities_generator, key=len, reverse=True)
    num_communities = len(communities)
    print(f"   ✅ 发现 {num_communities} 个主要社区岛屿，正在生成脉络...")

    # 分层布局：先将社区中心排布在圆环上，再在局部展开节点
    meta_pos = {}
    radius = 15.0 
    for i in range(num_communities):
        angle = (2 * math.pi * i) / num_communities
        meta_pos[i] = (radius * math.cos(angle), radius * math.sin(angle))

    pos = {}
    for i, comm in enumerate(communities):
        subgraph = G_undirected.subgraph(comm)
        sub_pos = nx.spring_layout(subgraph, k=0.15, iterations=20, seed=2026, center=meta_pos[i], scale=2.5)
        for node, (x, y) in sub_pos.items():
            pos[node] = (x, y)
    
    # 获取严格对齐的节点列表与颜色序列
    nodelist = list(G.nodes())
    node_colors = [color_map.get(node_types.get(node, 'default'), '#cccccc') for node in nodelist]
    
    # 绘制节点和边 (引入曲线脉络效果)
    nx.draw_networkx_nodes(G, pos, nodelist=nodelist, node_color=node_colors, node_size=18, alpha=0.9, edgecolors='none', ax=ax)
    nx.draw_networkx_edges(G, pos, alpha=0.15, edge_color='#7f8c8d', width=0.1, arrows=False, connectionstyle='arc3,rad=0.05', ax=ax)
    
    # ---------------- 4. 绘制左上角图例 ----------------
    legend_elements = []
    
    # 添加节点类型颜色图例
    for ntype, color in color_map.items():
        if ntype in node_type_counts: # 只显示图谱中实际存在的类型
            legend_elements.append(mpatches.Patch(color=color, label=f"{ntype} ({node_type_counts[ntype]})"))
        
    # 设置图例格式，放置在左上角
    legend = plt.legend(handles=legend_elements, loc='upper left', fontsize=16, 
                        frameon=True, facecolor='#2c3e50', edgecolor='white', labelcolor='white', title="Node Types", title_fontsize=18)
    legend.get_title().set_color("white")
    
    plt.title('AIBL-DementiaHKG Knowledge Graph', fontsize=30, color='white', fontweight='bold', pad=20)
    plt.axis('off')
    
    # ---------------- 5. 导出并保存为 PDF ----------------
    print(f"\n💾 正在导出矢量 PDF 文件...")
    plt.savefig(OUTPUT_PDF_PATH, format='pdf', bbox_inches='tight', facecolor='#1a1a1a')
    plt.close()
    
    print(f"   🎉 绘制成功！文件已保存至: {OUTPUT_PDF_PATH}")
    print("==================================================")

if __name__ == "__main__":
    plot_static_kg_pdf_with_stats()

📊 开始统计 AIBL-DementiaHKG 图谱数据并绘制静态高清 PDF...

📈 [整体规模统计]:
   - 节点总数: 3894
   - 边总数: 22610
   - 三元组总数: 22621

🧬 [各类节点数量统计]:
   - gene/protein: 2245
   - disease: 553
   - biological_process: 543
   - effect/phenotype: 157
   - anatomy: 146
   - cellular_component: 117
   - molecular_function: 82
   - pathway: 51

🔗 [各类关系数量统计]:
   - anatomy_protein_present: 9443
   - protein_protein: 3239
   - bioprocess_protein: 2834
   - disease_phenotype_positive: 2332
   - disease_protein: 2108
   - cellcomp_protein: 909
   - molfunc_protein: 620
   - pathway_protein: 368
   - phenotype_protein: 249
   - bioprocess_bioprocess: 220
   - phenotype_phenotype: 94
   - disease_disease: 67
   - anatomy_protein_absent: 50
   - disease_phenotype_negative: 36
   - molfunc_molfunc: 23
   - cellcomp_cellcomp: 18
   - anatomy_anatomy: 10
   - pathway_pathway: 1

⏳ 正在进行脉络化社区布局计算 (节点较多，可能需要 1-2 分钟，请耐心等待)...
   ✅ 发现 97 个主要社区岛屿，正在生成脉络...

💾 正在导出矢量 PDF 文件...
   🎉 绘制成功！文件已保存至: AIBL-DementiaHKGE.pdf


In [1]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import os
import random
import warnings

warnings.filterwarnings('ignore')

# ================= 配置区 =================
# 读取AIBL详细图谱数据
INPUT_KG_PATH = "AIBL_DementiaHKG_PrimeKG_Sub.csv"
OUTPUT_HTML_PATH = "AIBL_DementiaHKG_Diverse_Connected.html"

def plot_diverse_connected_graph():
    print("==================================================")
    print("🌐 开始构建AIBL多类型均衡的连通图谱 (白色背景, 高对比度配色)...")

    if not os.path.exists(INPUT_KG_PATH):
        print(f"❌ 找不到文件: {INPUT_KG_PATH}")
        return

    df = pd.read_csv(INPUT_KG_PATH)

    # ---------------- 1. 统计全量图谱节点并建立映射 ----------------
    print("📊 正在统计全量数据节点...")
    nodes_x = df[['x_name', 'x_type']].rename(columns={'x_name': 'name', 'x_type': 'type'})
    nodes_y = df[['y_name', 'y_type']].rename(columns={'y_name': 'name', 'y_type': 'type'})
    all_nodes = pd.concat([nodes_x, nodes_y]).drop_duplicates(subset=['name'])
    
    # 统计全量类型数量用于左上角图例展示
    full_type_counts = all_nodes['type'].value_counts().to_dict()
    node_type_map = dict(zip(all_nodes['name'], all_nodes['type']))

    # ---------------- 2. 提取多类型均衡连通子图 (Shortest Path Routing) ----------------
    print("⏳ 正在进行多类型路由寻径，确保包含所有颜色节点且不断开...")
    G_full = nx.from_pandas_edgelist(df, source='x_name', target='y_name', edge_attr='relation', create_using=nx.Graph())
    
    # 提取最大连通分量，保证基础连通性
    largest_cc = max(nx.connected_components(G_full), key=len)
    G_sub = G_full.subgraph(largest_cc)

    # 找到网络中度数最高的核心节点作为“根节点”
    central_node = sorted(G_sub.degree, key=lambda x: x[1], reverse=True)[0][0]

    # 将 LCC (最大连通分量) 中的节点按类型分组
    nodes_by_type = {}
    for node in G_sub.nodes():
        ntype = node_type_map.get(node, 'unknown')
        if ntype not in nodes_by_type:
            nodes_by_type[ntype] = []
        nodes_by_type[ntype].append(node)

    # 强制每种类型抽取最多 60 个目标节点
    random.seed(42) # 固定随机种子
    TARGET_PER_TYPE = 60
    core_nodes = set([central_node])

    for ntype, nodes_list in nodes_by_type.items():
        # 随机采样该类型的目标节点
        sampled_targets = random.sample(nodes_list, min(TARGET_PER_TYPE, len(nodes_list)))
        for target in sampled_targets:
            try:
                # 寻找中心节点到目标节点的最短路径
                path = nx.shortest_path(G_sub, source=central_node, target=target)
                core_nodes.update(path) # 将路径上的所有节点加入核心集合，保证连通不断开
            except nx.NetworkXNoPath:
                continue

    # 提取包含上述所有路径节点的子图 (同时会自动包含它们之间的所有内部边)
    G_core = G_sub.subgraph(core_nodes)

    # ---------------- 3. 高对比度配色方案 ----------------
    color_map = {
        'disease': '#E6194B',            # 鲜红
        'gene/protein': '#3CB44B',       # 翠绿
        'effect/phenotype': '#FFE119',   # 亮黄
        'anatomy': '#4363D8',            # 群青
        'biological_process': '#911EB4', # 深紫
        'pathway': '#F58231',            # 亮橙
        'cellular_component': '#42D4F4', # 青色
        'molecular_function': '#F032E6'  # 品红
    }

    # 初始化网络画布
    net = Network(height="900px", width="100%", bgcolor="#ffffff", font_color="#333333", select_menu=True)

    # ---------------- 4. 注入节点与边 (配置隐藏标签) ----------------
    for node in G_core.nodes():
        ntype = node_type_map.get(node, 'unknown')
        color = color_map.get(ntype, '#A9A9A9')
        
        # label 设为空格隐藏文字，title 用于悬停显示
        net.add_node(node, 
                     label=" ", 
                     title=f"实体名称: {node}\n实体类型: {ntype}",
                     color=color,
                     size=15)

    for u, v, data in G_core.edges(data=True):
        net.add_edge(u, v, 
                     title=data['relation'], 
                     label=' ', 
                     width=0.5,         
                     color='#bdc3c7')   # 浅灰色连线，突出节点颜色

    # 配置物理引擎使图形均匀展开
    net.force_atlas_2based(gravity=-50, central_gravity=0.01, spring_length=150, spring_strength=0.05, damping=0.4, overlap=0)
    
    net.write_html(OUTPUT_HTML_PATH)

    # ---------------- 5. 注入全局统计面板 ----------------
    legend_html = """
    <div style="position: absolute; top: 20px; left: 20px; z-index: 999; background-color: rgba(255, 255, 255, 0.95); color: #333; padding: 15px 25px; border-radius: 8px; border: 1px solid #ddd; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; box-shadow: 0 4px 10px rgba(0,0,0,0.1);">
        <h3 style="margin-top: 0; margin-bottom: 15px; font-size: 16px; border-bottom: 2px solid #eee; padding-bottom: 5px; color: #2c3e50;">AIBL全量图谱实体统计</h3>
    """
    
    for ntype, count in sorted(full_type_counts.items(), key=lambda x: x[1], reverse=True):
        color = color_map.get(ntype, '#A9A9A9')
        legend_html += f'<div style="margin: 8px 0; font-size: 13px; display: flex; align-items: center;"><span style="display: inline-block; width: 14px; height: 14px; background-color: {color}; border-radius: 50%; margin-right: 12px; border: 1px solid #888;"></span><span style="font-weight: 500; width: 150px;">{ntype}</span> <span>{count:,}</span></div>'
        
    legend_html += "</div>"

    with open(OUTPUT_HTML_PATH, 'r', encoding='utf-8') as f:
        html_content = f.read()
    
    html_content = html_content.replace('<body>', f'<body>\n{legend_html}')
    
    with open(OUTPUT_HTML_PATH, 'w', encoding='utf-8') as f:
        f.write(html_content)

    print(f"💾 AIBL多类型均衡图谱已生成，文件保存至: {OUTPUT_HTML_PATH}")
    print("==================================================")

if __name__ == "__main__":
    plot_diverse_connected_graph()

🌐 开始构建AIBL多类型均衡的连通图谱 (白色背景, 高对比度配色)...
📊 正在统计全量数据节点...
⏳ 正在进行多类型路由寻径，确保包含所有颜色节点且不断开...
💾 AIBL多类型均衡图谱已生成，文件保存至: AIBL_DementiaHKG_Diverse_Connected.html
